# LeetCode 1400 — Construct K Palindrome Strings bằng Python

Link đề: https://leetcode.com/problems/construct-k-palindrome-strings/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Greedy, Counting

Skeleton LeetCode:

```python
class Solution:
    def canConstruct(self, s: str, k: int) -> bool:
        ...
```

## 1. Mục tiêu học

1. Hiểu **điều kiện để một nhóm ký tự xếp được thành palindrome**: nhiều nhất **một** ký tự xuất hiện lẻ lần.
2. Từ đó rút ra tiêu chí cho `k` palindrome: `số ký tự lẻ ≤ k ≤ len(s)` — và **chứng minh** cả hai chiều (cần và đủ, có cách dựng cụ thể).
3. Biết đi từ brute force (thử mọi cách chia ký tự vào `k` nhóm) tới đếm bằng `Counter`, mảng 26 phần tử, và **bitmask chẵn lẻ** với XOR.
4. Dùng brute force trên chuỗi nhỏ để **kiểm chứng** một công thức "nghe có vẻ đúng".
5. Tránh lỗi: quên điều kiện `k ≤ len(s)`, dùng `==` thay `≤`, đếm ký tự "xuất hiện 1 lần" thay vì "xuất hiện lẻ lần".

## 2. Đề bài & ví dụ

Cho chuỗi `s` và số nguyên `k`. Trả về `True` nếu có thể dùng **tất cả** ký tự của `s` (mỗi ký tự dùng đúng một lần, xếp lại tuỳ ý) để tạo **đúng `k` chuỗi palindrome khác rỗng**; ngược lại `False`.

**Ví dụ 1:** `s = "annabelle", k = 2` → `True`. Một số cách: `"anna" + "elble"`, `"anbna" + "elle"`, `"anellena" + "b"`.

**Ví dụ 2:** `s = "leetcode", k = 3` → `False`.

**Ví dụ 3:** `s = "true", k = 4` → `True` (mỗi ký tự thành một chuỗi riêng).

**Ràng buộc:** `1 <= s.length <= 10^5`, `s` gồm chữ cái thường tiếng Anh, `1 <= k <= 10^5`.

In [1]:
import random
import string
import time
from collections import Counter
from itertools import product
from typing import List

examples = [("annabelle", 2, True), ("leetcode", 3, False), ("true", 4, True)]
for s, k, want in examples:
    print(f"s={s!r}, k={k} -> {want}")

s='annabelle', k=2 -> True
s='leetcode', k=3 -> False
s='true', k=4 -> True


## 3. Kiến thức nền

### 3.1 Palindrome và kiểm tra bằng đảo chuỗi

Palindrome là chuỗi đọc xuôi ngược như nhau. Trong Python: `t == t[::-1]`.

In [2]:
for t in ["anna", "elble", "abc", "a", "abba"]:
    print(t, t == t[::-1])

anna True
elble True
abc False
a True
abba True


### 3.2 `collections.Counter` đếm tần suất

`Counter(s)` trả về `dict` ký tự → số lần xuất hiện, `O(n)`.

In [3]:
cnt = Counter("annabelle")
print(cnt)
print("ký tự xuất hiện lẻ lần:", [ch for ch, v in cnt.items() if v % 2 == 1])

Counter({'a': 2, 'n': 2, 'e': 2, 'l': 2, 'b': 1})
ký tự xuất hiện lẻ lần: ['b']


### 3.3 Khi nào một tập ký tự xếp được thành palindrome?

Palindrome đối xứng quanh tâm: mỗi ký tự ở nửa trái có một bản sao ở nửa phải. Vậy mọi ký tự xuất hiện **chẵn** lần, **trừ** có thể một ký tự nằm ở chính giữa (khi độ dài lẻ). Kết luận: **xếp được ⇔ số ký tự có tần suất lẻ ≤ 1**.

In [4]:
def can_be_palindrome(chars):
    return sum(v % 2 for v in Counter(chars).values()) <= 1

for t in ["aabb", "aab", "abc", "aaabbbb", "z"]:
    print(t, can_be_palindrome(t))

aabb True
aab True
abc False
aaabbbb True
z True


### 3.4 `v % 2` và `v & 1`

Cả hai đều cho `1` nếu `v` lẻ, `0` nếu chẵn. `sum(v % 2 for ...)` đếm số giá trị lẻ.

In [5]:
vals = [4, 1, 3, 2, 7]
print([v % 2 for v in vals], [v & 1 for v in vals], "số lẻ:", sum(v & 1 for v in vals))

[0, 1, 1, 0, 1] [0, 1, 1, 0, 1] số lẻ: 3


### 3.5 Mảng 26 phần tử với `ord`

`ord(ch) - ord('a')` đổi chữ thường thành chỉ số `0..25`. Dùng list cố định thay cho `Counter` khi chỉ có chữ thường.

In [6]:
freq = [0] * 26
for ch in "leetcode":
    freq[ord(ch) - ord('a')] += 1
print({chr(i + 97): f for i, f in enumerate(freq) if f})

{'c': 1, 'd': 1, 'e': 3, 'l': 1, 'o': 1, 't': 1}


### 3.6 Bitmask chẵn lẻ với XOR

Gán mỗi chữ một bit. `mask ^= 1 << idx` **lật** bit đó mỗi khi gặp chữ. Cuối cùng bit bật ⇔ chữ xuất hiện lẻ lần. Số bit bật: `bin(mask).count("1")` (hoặc `mask.bit_count()` từ Python 3.10).

In [7]:
mask = 0
for ch in "annabelle":
    mask ^= 1 << (ord(ch) - 97)
print(f"mask = {mask:026b}")
print("số chữ lẻ:", bin(mask).count("1"))

mask = 00000000000000000000000010
số chữ lẻ: 1


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 2** `s = "leetcode"`, `k = 3`:

- Tần suất: `l:1, e:3, t:1, c:1, o:1, d:1` ⇒ các chữ lẻ: `l, e, t, c, o, d` — **6** chữ lẻ.
- Mỗi palindrome chứa **tối đa một** chữ lẻ ở giữa. Với 3 palindrome, nhiều nhất "hấp thụ" được 3 chữ lẻ. Còn 3 chữ lẻ dư ⇒ **không thể** ⇒ `False`.

**Ví dụ 1** `"annabelle"`, `k = 2`: `a:2, n:2, b:1, e:2, l:2` ⇒ 1 chữ lẻ (`b`). 1 ≤ 2 ⇒ có hi vọng. Và `len = 9 ≥ 2` nên tách được thành 2 chuỗi khác rỗng ⇒ `True`.

**Ví dụ 3** `"true"`, `k = 4`: 4 chữ lẻ, `k = 4`, `len = 4` ⇒ mỗi chữ một chuỗi ⇒ `True`.

Nhận xét: có hai rào cản — (1) **quá nhiều chữ lẻ** so với `k`; (2) **quá ít ký tự** để tạo `k` chuỗi khác rỗng.

In [8]:
for s, k, want in examples:
    odd = sum(v % 2 for v in Counter(s).values())
    print(f"{s!r}: len={len(s)}, số chữ lẻ={odd}, k={k} -> odd<=k<=len? {odd <= k <= len(s)} (đề: {want})")

'annabelle': len=9, số chữ lẻ=1, k=2 -> odd<=k<=len? True (đề: True)
'leetcode': len=8, số chữ lẻ=6, k=3 -> odd<=k<=len? False (đề: False)
'true': len=4, số chữ lẻ=4, k=4 -> odd<=k<=len? True (đề: True)


## 5. Cách 1 — Brute Force / baseline: thử mọi cách chia ký tự vào `k` nhóm

### Ý tưởng
Chia các **vị trí** của `s` thành đúng `k` nhóm khác rỗng (không phân biệt thứ tự nhóm), theo kiểu đệ quy: ký tự thứ `i` hoặc vào một nhóm đã có, hoặc mở nhóm mới (nếu chưa đủ `k` nhóm). Với mỗi phép chia, kiểm tra **mọi nhóm xếp được thành palindrome** (số chữ lẻ ≤ 1). Số phép chia là số Stirling loại 2 `S(n, k)` — tăng theo hàm mũ, chỉ chạy được với `n` nhỏ, nhưng **không dựa vào công thức nào** ⇒ dùng làm thước đo.

In [9]:
def can_construct_brute(s, k):
    n = len(s)
    if k > n:
        return False                       # không đủ ký tự cho k nhóm khác rỗng
    groups = []

    def rec(i):
        if k - len(groups) > n - i:        # không đủ ký tự còn lại để mở đủ k nhóm
            return False
        if i == n:
            return all(can_be_palindrome(g) for g in groups)
        for g in groups:                   # cho s[i] vào một nhóm đã có
            g.append(s[i])
            ok = rec(i + 1)
            g.pop()
            if ok:
                return True
        if len(groups) < k:                # hoặc mở nhóm mới
            groups.append([s[i]])
            ok = rec(i + 1)
            groups.pop()
            if ok:
                return True
        return False

    return rec(0)

for s, k, want in [("true", 4, True), ("aabb", 2, True), ("abc", 2, False), ("abcab", 2, True)]:
    print(f"{s!r}, k={k}: {can_construct_brute(s, k)} (mong đợi {want})")

'true', k=4: True (mong đợi True)
'aabb', k=2: True (mong đợi True)
'abc', k=2: False (mong đợi False)
'abcab', k=2: True (mong đợi True)


Ví dụ trong đề có `n = 8, 9` và `k` nhỏ: `S(9, 2) = 255`, `S(8, 3) = 966` phép chia — vẫn chạy được:

In [10]:
for s, k, want in examples:
    t = time.perf_counter()
    res = can_construct_brute(s, k)
    print(f"{s!r}, k={k}: {res} (mong đợi {want}) — {time.perf_counter() - t:.3f}s")

'annabelle', k=2: True (mong đợi True) — 0.000s
'leetcode', k=3: False (mong đợi False) — 0.001s
'true', k=4: True (mong đợi True) — 0.000s


### Độ phức tạp và nút thắt

- Time `O(S(n,k) · n)` ≤ `O(k^n · n)`. Với `n = 10^5` là điều không tưởng.
- Space `O(n)`.

Nút thắt: ta quan tâm **từng vị trí ký tự**, trong khi palindrome chỉ phụ thuộc vào **số lượng** mỗi chữ — thậm chí chỉ **tính chẵn lẻ** của số lượng.

## 6. Ý tưởng thuật toán / bước đột phá

Gọi `odd` = số chữ có tần suất lẻ, `n = len(s)`.

**Điều kiện cần 1 — `k ≤ n`:** mỗi palindrome khác rỗng cần ≥ 1 ký tự.

**Điều kiện cần 2 — `odd ≤ k`:** một chữ có tần suất lẻ, khi chia vào các nhóm, phải nằm lẻ lần ở **ít nhất một** nhóm (tổng các số chẵn là chẵn). Mỗi palindrome chứa tối đa một chữ lẻ ⇒ cần ít nhất `odd` palindrome.

**Điều kiện đủ — nếu `odd ≤ k ≤ n` thì dựng được** (chứng minh chi tiết ở mục 10): đặt mỗi chữ lẻ làm tâm một palindrome riêng (`odd` chuỗi), phần còn lại toàn **cặp** ký tự; tạo thêm `k − odd` chuỗi bằng cách tách từng ký tự lẻ từ các cặp… và mọi thứ dư thì gắn đối xứng vào một palindrome có sẵn.

**Quy tắc "nếu… thì…":**
- Nếu `k > n` ⇒ `False`.
- Nếu `odd > k` ⇒ `False`.
- Ngược lại ⇒ `True`.

Đáp án = `odd <= k <= len(s)`.

## 7. Các cách tiếp theo

### Cách 2 — Đếm bằng `Counter`

**Từng đoạn:**
1. Nếu `k > len(s)` trả `False` ngay (rẻ, khỏi đếm).
2. `odd = sum(v % 2 for v in Counter(s).values())`.
3. Trả `odd <= k`.

In [11]:
def can_construct_counter(s, k):
    if k > len(s):
        return False
    odd = sum(v % 2 for v in Counter(s).values())
    return odd <= k

for s, k, want in examples:
    print(can_construct_counter(s, k), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
True | mong đợi True


In [12]:
class SolutionCounter:
    def canConstruct(self, s: str, k: int) -> bool:
        if k > len(s):
            return False
        return sum(v & 1 for v in Counter(s).values()) <= k

print([SolutionCounter().canConstruct(s, k) for s, k, _ in examples])

[True, False, True]


Time `O(n)`, Space `O(σ)` với `σ = 26` (số chữ cái) ⇒ `O(1)`.

### Cách 3 — Mảng tần suất 26 phần tử

Như Cách 2 nhưng tránh `Counter` (hữu ích khi viết bằng ngôn ngữ khác, hoặc muốn rõ ràng về bộ nhớ hằng số).

In [13]:
def can_construct_array(s, k):
    if k > len(s):
        return False
    freq = [0] * 26
    for ch in s:
        freq[ord(ch) - 97] += 1
    odd = 0
    for f in freq:
        odd += f & 1
    return odd <= k

for s, k, want in examples:
    print(can_construct_array(s, k), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
True | mong đợi True


Time `O(n + 26)`, Space `O(26)`.

### Cách 4 — Bitmask chẵn lẻ (chỉ giữ tính chẵn lẻ)

Ta không cần số lượng, chỉ cần **chẵn hay lẻ** ⇒ một số nguyên 26 bit đủ. Mỗi ký tự lật một bit; cuối cùng đếm số bit 1.

In [14]:
def can_construct_mask(s, k):
    if k > len(s):
        return False
    mask = 0
    for ch in s:
        mask ^= 1 << (ord(ch) - 97)
    return bin(mask).count("1") <= k

for s, k, want in examples:
    print(can_construct_mask(s, k), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
True | mong đợi True


In [15]:
class SolutionMask:
    def canConstruct(self, s: str, k: int) -> bool:
        if k > len(s):
            return False
        mask = 0
        for ch in s:
            mask ^= 1 << (ord(ch) - 97)
        return bin(mask).count("1") <= k

print([SolutionMask().canConstruct(s, k) for s, k, _ in examples])

[True, False, True]


Time `O(n)`, Space `O(1)` thực sự (một số nguyên). Kỹ thuật bitmask chẵn lẻ là chìa khoá cho các biến thể có **truy vấn đoạn con** (prefix XOR — Bài 5 mục 17).

### Cách 5 — Dựng hẳn `k` palindrome (cách "xây dựng")

Không chỉ trả lời có/không mà **tạo ra** `k` chuỗi. Đây cũng là bằng chứng cho chiều "đủ".

**Thuật toán dựng** (khi `odd ≤ k ≤ n`):
1. Mỗi chữ lẻ: lấy ra **một** bản làm tâm của một palindrome mới ⇒ `odd` palindrome độ dài 1. Phần còn lại của mọi chữ đều **chẵn** ⇒ gom thành danh sách **cặp**.
2. Còn thiếu `need = k − odd` palindrome: mỗi lần **tách một cặp** `xx` thành hai palindrome `"x"` và `"x"` (tạo thêm 2), hoặc nếu chỉ thiếu 1 thì lấy một cặp làm palindrome `"xx"` (tạo thêm 1).
3. Các cặp còn dư: bọc đối xứng quanh palindrome đầu tiên (`x + p + x`) — vẫn là palindrome.

Vì `n − odd` là chẵn và `k ≤ n`, luôn đủ cặp cho bước 2 (mục 10 chứng minh).

In [16]:
def build_palindromes(s, k):
    """Trả về list k palindrome khác rỗng dùng hết ký tự của s, hoặc None nếu không thể."""
    if k > len(s):
        return None
    cnt = Counter(s)
    centers = [ch for ch, v in cnt.items() if v % 2 == 1]
    if len(centers) > k:
        return None
    pals = list(centers)                                  # bước 1: mỗi chữ lẻ một tâm
    pairs = []
    for ch, v in cnt.items():
        pairs.extend([ch] * (v // 2))                     # các cặp còn lại
    need = k - len(pals)
    while need >= 2:                                      # bước 2: tách cặp thành 2 chuỗi
        ch = pairs.pop()
        pals += [ch, ch]
        need -= 2
    if need == 1:
        ch = pairs.pop()
        pals.append(ch + ch)
    for ch in pairs:                                      # bước 3: bọc cặp dư quanh pals[0]
        pals[0] = ch + pals[0] + ch
    return pals

for s, k, _ in examples + [("aabbccd", 5, True)]:
    print(f"{s!r}, k={k} ->", build_palindromes(s, k))

'annabelle', k=2 -> ['enabane', 'll']
'leetcode', k=3 -> None
'true', k=4 -> ['t', 'r', 'u', 'e']
'aabbccd', k=5 -> ['ada', 'c', 'c', 'b', 'b']


Time `O(n²)` trong trường hợp xấu do nối chuỗi lặp ở bước 3 (có thể sửa thành `O(n)` bằng cách gom nửa trái rồi ghép một lần); Space `O(n)`. Dùng để **minh hoạ và kiểm chứng**, không cần cho bài nộp.

## 8. TRACE từng bước

Trace Cách 4 (bitmask) trên **Ví dụ 2** `"leetcode"` — in mask và số chữ lẻ sau mỗi ký tự:

In [17]:
def trace_mask(s, k):
    print(f"s={s!r}, k={k}, len={len(s)}")
    if k > len(s):
        print("k > len(s) -> False"); return False
    mask = 0
    for i, ch in enumerate(s):
        mask ^= 1 << (ord(ch) - 97)
        odd_chars = [chr(97 + b) for b in range(26) if mask >> b & 1]
        print(f"  i={i} ch={ch!r}: chữ đang lẻ = {odd_chars} (số lẻ = {len(odd_chars)})")
    odd = bin(mask).count("1")
    print(f"  kết thúc: odd={odd} {'<=' if odd <= k else '>'} k={k} -> {odd <= k}")
    return odd <= k

trace_mask("leetcode", 3)

s='leetcode', k=3, len=8
  i=0 ch='l': chữ đang lẻ = ['l'] (số lẻ = 1)
  i=1 ch='e': chữ đang lẻ = ['e', 'l'] (số lẻ = 2)
  i=2 ch='e': chữ đang lẻ = ['l'] (số lẻ = 1)
  i=3 ch='t': chữ đang lẻ = ['l', 't'] (số lẻ = 2)
  i=4 ch='c': chữ đang lẻ = ['c', 'l', 't'] (số lẻ = 3)
  i=5 ch='o': chữ đang lẻ = ['c', 'l', 'o', 't'] (số lẻ = 4)
  i=6 ch='d': chữ đang lẻ = ['c', 'd', 'l', 'o', 't'] (số lẻ = 5)
  i=7 ch='e': chữ đang lẻ = ['c', 'd', 'e', 'l', 'o', 't'] (số lẻ = 6)
  kết thúc: odd=6 > k=3 -> False


False

| i | ch | chữ đang lẻ | số lẻ |
|---|---|---|---|
| 0 | l | l | 1 |
| 1 | e | l, e | 2 |
| 2 | e | l | 1 |
| 3 | t | l, t | 2 |
| 4 | c | c, l, t | 3 |
| 5 | o | c, l, o, t | 4 |
| 6 | d | c, d, l, o, t | 5 |
| 7 | e | c, d, e, l, o, t | 6 |

6 > 3 ⇒ `False`.

Trace Cách 5 (dựng) trên `s = "annabelle"`, `k = 4` — in từng bước dựng:

In [18]:
def trace_build(s, k):
    cnt = Counter(s)
    print("tần suất:", dict(cnt))
    centers = [ch for ch, v in cnt.items() if v % 2]
    pals = list(centers)
    pairs = [ch for ch, v in cnt.items() for _ in range(v // 2)]
    print("bước 1: tâm =", pals, "| cặp =", pairs)
    need = k - len(pals)
    while need >= 2:
        ch = pairs.pop(); pals += [ch, ch]; need -= 2
        print(f"bước 2: tách cặp {ch}{ch} -> pals = {pals}, còn thiếu {need}")
    if need == 1:
        ch = pairs.pop(); pals.append(ch + ch)
        print(f"bước 2': thêm '{ch}{ch}' -> pals = {pals}")
    for ch in pairs:
        pals[0] = ch + pals[0] + ch
        print(f"bước 3: bọc {ch} quanh pals[0] -> {pals}")
    return pals

res = trace_build("annabelle", 4)
print("kết quả:", res, "| đều là palindrome?", all(p == p[::-1] for p in res))

tần suất: {'a': 2, 'n': 2, 'b': 1, 'e': 2, 'l': 2}
bước 1: tâm = ['b'] | cặp = ['a', 'n', 'e', 'l']
bước 2: tách cặp ll -> pals = ['b', 'l', 'l'], còn thiếu 1
bước 2': thêm 'ee' -> pals = ['b', 'l', 'l', 'ee']
bước 3: bọc a quanh pals[0] -> ['aba', 'l', 'l', 'ee']
bước 3: bọc n quanh pals[0] -> ['naban', 'l', 'l', 'ee']
kết quả: ['naban', 'l', 'l', 'ee'] | đều là palindrome? True


## 9. Code nộp LeetCode

In [19]:
from collections import Counter

class Solution:
    def canConstruct(self, s: str, k: int) -> bool:
        if k > len(s):                       # không đủ ký tự cho k chuỗi khác rỗng
            return False
        odd = sum(v & 1 for v in Counter(s).values())
        return odd <= k                      # mỗi palindrome "hấp thụ" tối đa 1 chữ lẻ

for s, k, want in examples:
    assert Solution().canConstruct(s, k) == want
print("Solution đúng với 3 ví dụ của đề")

Solution đúng với 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

Ký hiệu `n = len(s)`, `odd` = số chữ có tần suất lẻ. Khẳng định: **dựng được ⇔ `odd ≤ k ≤ n`**.

**(⇒) Cần.**
- `k ≤ n`: `k` chuỗi khác rỗng, rời nhau, dùng hết `n` ký tự ⇒ `k ≤ n`.
- `odd ≤ k`: xét chữ `c` có tần suất lẻ. Tần suất của `c` = tổng tần suất của `c` trong `k` chuỗi. Nếu mọi số hạng đều chẵn thì tổng chẵn — mâu thuẫn. Vậy có ít nhất một chuỗi chứa `c` lẻ lần. Mỗi palindrome có tối đa một chữ lẻ ⇒ các chữ lẻ khác nhau "chiếm" các palindrome khác nhau ⇒ `odd ≤ k`.

**(⇐) Đủ** — chính là thuật toán dựng ở Cách 5:
- Sau bước 1 có `odd` palindrome độ dài 1 và `P = (n − odd)/2` cặp (vì `n − odd` chẵn: mỗi chữ lẻ bớt 1 bản thì thành chẵn).
- Bước 2 cần tạo thêm `need = k − odd` chuỗi. Mỗi cặp tạo thêm tối đa 2 chuỗi ⇒ cần `⌈need/2⌉` cặp. Vì `k ≤ n` ⇒ `need ≤ n − odd = 2P` ⇒ `⌈need/2⌉ ≤ P` ✔ đủ cặp.
- Bước 3: bọc `x … x` quanh một palindrome vẫn là palindrome; nếu `k ≥ 1` luôn có `pals[0]`.
- Kết quả: đúng `k` chuỗi, đều palindrome, khác rỗng, dùng hết ký tự.

Phần kiểm chứng: mục 12 so công thức với brute force trên mọi chuỗi nhỏ, và kiểm tra `build_palindromes` luôn cho kết quả hợp lệ khi công thức nói `True`.

## 11. Phân tích Time / Space

Gọi `n = len(s)` (≤ 10⁵), `σ = 26`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force chia nhóm | `O(k^n · n)` | `O(n)` |
| 2. `Counter` | `O(n)` | `O(σ)` |
| 3. Mảng 26 | `O(n + σ)` | `O(σ)` |
| 4. Bitmask | `O(n)` | `O(1)` |
| 5. Dựng palindrome | `O(n²)` (nối chuỗi) — có thể `O(n)` | `O(n)` |

Khi `k > n`, Cách 2–4 trả về ngay `O(1)` trước khi đếm.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [20]:
ALL = {
    "counter": can_construct_counter,
    "counter_submit": lambda s, k: SolutionCounter().canConstruct(s, k),
    "array": can_construct_array,
    "mask": can_construct_mask,
    "mask_submit": lambda s, k: SolutionMask().canConstruct(s, k),
    "build": lambda s, k: build_palindromes(s, k) is not None,
    "submit": lambda s, k: Solution().canConstruct(s, k),
}
edge = [
    ("a", 1, True), ("a", 2, False),          # n = 1
    ("ab", 1, False), ("ab", 2, True),         # 2 chữ lẻ
    ("ab", 3, False),                          # k > n dù odd <= k
    ("aa", 1, True), ("aa", 2, True),          # tách cặp
    ("aaa", 2, True),                          # "a" + "aa"
    ("abc", 2, False), ("abc", 3, True),
    ("aabbcc", 1, True), ("aabbcc", 6, True), ("aabbcc", 5, True),
    ("aabbccd", 1, True), ("aabbcde", 2, False),
]
for s, k, want in edge:
    assert can_construct_brute(s, k) == want, (s, k)
for name, f in ALL.items():
    for s, k, want in examples + edge:
        assert f(s, k) == want, (name, s, k)
print("Mọi cách đúng với", len(examples + edge), "test (nhãn edge kiểm chứng bằng brute force)")

Mọi cách đúng với 18 test (nhãn edge kiểm chứng bằng brute force)


### 12.2 Kiểm chứng công thức bằng brute force trên **mọi** chuỗi nhỏ

Duyệt **toàn bộ** chuỗi độ dài 1..6 trên bảng chữ `{a, b, c}` và mọi `k` từ 1 đến 7 — không bỏ sót trường hợp nào. Đồng thời kiểm tra `build_palindromes` cho kết quả hợp lệ.

In [21]:
def valid_construction(s, k, pals):
    return (pals is not None and len(pals) == k and all(pals)
            and all(p == p[::-1] for p in pals)
            and Counter("".join(pals)) == Counter(s))

checked = 0
for n in range(1, 7):
    for tup in product("abc", repeat=n):
        s = "".join(tup)
        for k in range(1, 8):
            want = can_construct_brute(s, k)
            for name, f in ALL.items():
                assert f(s, k) == want, (name, s, k)
            if want:
                assert valid_construction(s, k, build_palindromes(s, k)), (s, k)
            checked += 1
print("đã kiểm tra", checked, "cặp (s, k): công thức khớp brute force, phép dựng luôn hợp lệ")

đã kiểm tra 7644 cặp (s, k): công thức khớp brute force, phép dựng luôn hợp lệ


### 12.3 Ngẫu nhiên lớn hơn (seed cố định) + giới hạn `n = 10^5`

Brute force không chạy nổi nữa ⇒ kiểm tra **chéo** giữa các cách, và kiểm tra **phép dựng hợp lệ** (đây là bằng chứng "True" độc lập với công thức).

In [22]:
rng = random.Random(1400)
for _ in range(2000):
    n = rng.randint(1, 60)
    alpha = string.ascii_lowercase[: rng.randint(1, 26)]
    s = "".join(rng.choice(alpha) for _ in range(n))
    k = rng.randint(1, 70)
    res = {name: f(s, k) for name, f in ALL.items()}
    assert len(set(res.values())) == 1, (s, k, res)
    if res["submit"]:
        assert valid_construction(s, k, build_palindromes(s, k))
print("2000 test ngẫu nhiên: mọi cách thống nhất, phép dựng hợp lệ")

big = "".join(rng.choice(string.ascii_lowercase) for _ in range(10**5))
for name in ["counter", "array", "mask", "submit"]:
    t = time.perf_counter()
    r = [ALL[name](big, k) for k in (1, 26, 10**5)]
    print(f"{name:8s}: {r} ({time.perf_counter() - t:.3f}s)")

2000 test ngẫu nhiên: mọi cách thống nhất, phép dựng hợp lệ
counter : [False, True, True] (0.008s)
array   : [False, True, True] (0.012s)
mask    : [False, True, True] (0.016s)
submit  : [False, True, True] (0.009s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên điều kiện `k ≤ len(s)`

In [23]:
def wrong_no_length_check(s, k):
    odd = sum(v % 2 for v in Counter(s).values())
    return odd <= k                                   # SAI: thiếu k <= len(s)

for s, k in [("ab", 3), ("true", 5), ("a", 100000)]:
    print(f"{s!r}, k={k} | sai: {wrong_no_length_check(s, k)} | đúng: {Solution().canConstruct(s, k)}")

'ab', k=3 | sai: True | đúng: False
'true', k=5 | sai: True | đúng: False
'a', k=100000 | sai: True | đúng: False


`"ab"` chỉ có 2 ký tự, không thể tạo 3 chuỗi khác rỗng. **Sửa:** `if k > len(s): return False`.

### Lỗi 2 — Dùng `odd == k` thay vì `odd <= k`

Nghĩ rằng "mỗi palindrome phải có đúng một chữ lẻ". Thực ra palindrome độ dài chẵn (`"elle"`) không có chữ lẻ nào.

In [24]:
def wrong_equal(s, k):
    if k > len(s):
        return False
    odd = sum(v % 2 for v in Counter(s).values())
    return odd == k                                   # SAI

for s, k in [("annabelle", 2), ("aabb", 1), ("aa", 2)]:
    print(f"{s!r}, k={k} | sai: {wrong_equal(s, k)} | đúng: {Solution().canConstruct(s, k)}",
          "| ví dụ dựng:", build_palindromes(s, k))

'annabelle', k=2 | sai: False | đúng: True | ví dụ dựng: ['enabane', 'll']
'aabb', k=1 | sai: False | đúng: True | ví dụ dựng: ['abba']
'aa', k=2 | sai: False | đúng: True | ví dụ dựng: ['a', 'a']


**Sửa:** `odd <= k`.

### Lỗi 3 — Đếm chữ "xuất hiện đúng 1 lần" thay vì "xuất hiện lẻ lần"

In [25]:
def wrong_count_once(s, k):
    if k > len(s):
        return False
    once = sum(1 for v in Counter(s).values() if v == 1)   # SAI: bỏ sót tần suất 3, 5, ...
    return once <= k

for s, k in [("leetcode", 3), ("aaabbbccc", 1)]:
    print(f"{s!r}, k={k} | sai: {wrong_count_once(s, k)} | đúng: {Solution().canConstruct(s, k)}")

'leetcode', k=3 | sai: False | đúng: False
'aaabbbccc', k=1 | sai: True | đúng: False


`"leetcode"` có `e` xuất hiện 3 lần (lẻ) — bản sai chỉ đếm 5 chữ xuất hiện 1 lần ⇒ vẫn ra False tình cờ ở `k = 3`, nhưng `"aaabbbccc"` (ba chữ đều xuất hiện 3 lần) bản sai đếm 0 ⇒ `True`, đúng ra `False`. **Sửa:** `v % 2 == 1`.

### Lỗi 4 — Áp dụng điều kiện của **một** palindrome (`odd ≤ 1`)

In [26]:
def wrong_single_palindrome(s, k):
    if k > len(s):
        return False
    return sum(v % 2 for v in Counter(s).values()) <= 1      # SAI: quên vai trò của k

for s, k in [("true", 4), ("abc", 3)]:
    print(f"{s!r}, k={k} | sai: {wrong_single_palindrome(s, k)} | đúng: {Solution().canConstruct(s, k)}")

'true', k=4 | sai: False | đúng: True
'abc', k=3 | sai: False | đúng: True


**Sửa:** có `k` palindrome thì chứa được tới `k` chữ lẻ.

### Lỗi 5 — Bitmask dùng OR thay vì XOR

`mask |= bit` chỉ ghi nhận "đã xuất hiện", không lật được ⇒ đếm số **chữ khác nhau**, không phải số chữ lẻ.

In [27]:
def wrong_or_mask(s, k):
    if k > len(s):
        return False
    mask = 0
    for ch in s:
        mask |= 1 << (ord(ch) - 97)                          # SAI: phải là ^=
    return bin(mask).count("1") <= k

for s, k in [("annabelle", 2), ("aabb", 1)]:
    print(f"{s!r}, k={k} | sai: {wrong_or_mask(s, k)} | đúng: {Solution().canConstruct(s, k)}")

'annabelle', k=2 | sai: False | đúng: True
'aabb', k=1 | sai: False | đúng: True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Xếp lại ký tự thành palindrome"** ⇒ chỉ quan tâm **tính chẵn lẻ** của tần suất.
- "Được xếp lại tuỳ ý" ⇒ thứ tự ban đầu vô nghĩa ⇒ đếm tần suất.
- Câu hỏi có/không về khả năng chia ⇒ tìm **chặn dưới** và **chặn trên** cho `k` (ở đây `odd ≤ k ≤ n`), rồi chứng minh mọi giá trị ở giữa đều đạt được.
- Bảng chữ nhỏ (26) ⇒ bitmask XOR; có truy vấn đoạn con ⇒ prefix XOR.
- Gặp công thức "nghe có vẻ đúng" ⇒ viết brute force cho chuỗi nhỏ và **duyệt toàn bộ** để kiểm chứng.

## 15. Biến thể của bài

1. **Một palindrome** (LeetCode 266 *Palindrome Permutation*): `odd ≤ 1`.
2. **Palindrome dài nhất từ các ký tự** (LeetCode 409 *Longest Palindrome*): `n − odd + (1 if odd else 0)`.
3. **Truy vấn trên đoạn con** `s[l..r]` với `k` lần sửa ký tự (LeetCode 1177 *Can Make Palindrome from Substring*): prefix XOR mask, điều kiện `odd // 2 ≤ k`.
4. **Dựng hẳn** `k` palindrome (Bài 4 mục 17), hoặc dựng sao cho palindrome **dài nhất ngắn nhất** có thể.
5. **Khoảng `k` hợp lệ**: tập các `k` dựng được là đoạn liên tục `[max(1, odd), n]`.
6. Đếm số **đoạn con "wonderful"** (LeetCode 1915): tối đa một chữ lẻ — prefix mask + hash map.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi phép chia ký tự vào `k` nhóm | `O(k^n·n)` | `O(n)` | kiểm chứng công thức trên chuỗi nhỏ |
| 2. `Counter` | `odd ≤ k ≤ n` | `O(n)` | `O(σ)` | **mặc định**, ngắn gọn |
| 3. Mảng 26 | như trên, mảng cố định | `O(n)` | `O(26)` | ngôn ngữ không có `Counter` |
| 4. Bitmask XOR | chỉ giữ chẵn lẻ | `O(n)` | `O(1)` | biến thể truy vấn đoạn con |
| 5. Dựng palindrome | tâm = chữ lẻ, tách/bọc cặp | `O(n)`–`O(n²)` | `O(n)` | cần xuất kết quả, chứng minh chiều đủ |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().canConstruct("aabbc", 2)` và `Solution().canConstruct("aabbc", 6)` trả về gì?

### Bài 2 (tính tay)
Với `s = "annabelle"`, tìm **mọi** giá trị `k` để `canConstruct(s, k)` là `True`.

### Bài 3 (điền TODO)
Hoàn thiện `count_odd(s)`: trả về số chữ có tần suất lẻ, dùng mảng 26 phần tử.

### Bài 4 (tự viết lại)
Viết `build_k(s, k)` của riêng bạn trả về `k` palindrome (hoặc `None`), chạy `O(n)` (không nối chuỗi lặp nhiều lần).

### Bài 5 (mini challenge)
Cho `s` và nhiều truy vấn `(l, r, k)`: hỏi có thể dùng hết ký tự của `s[l..r]` để dựng `k` palindrome không? Viết `answer_queries(s, queries)` với tiền xử lý `O(n)` và mỗi truy vấn `O(1)` (coi đếm bit là `O(1)`).

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_odd(s):
    freq = [0] * 26
    # TODO: đếm tần suất từng chữ
    # TODO: trả về số phần tử lẻ trong freq
    pass

# Bài 4 — TODO
def build_k(s, k):
    # TODO: kiểm tra odd <= k <= len(s)
    # TODO: chọn tâm, chia cặp, gom nửa trái của palindrome đầu bằng list rồi ghép một lần
    pass

# Bài 5 — TODO
def answer_queries(s, queries):
    # TODO: pre[i] = XOR mask của s[:i]; mask đoạn [l, r] = pre[r+1] ^ pre[l]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tần suất `a:2, b:2, c:1` ⇒ `odd = 1`. (2) `k = 2`: `1 ≤ 2 ≤ 5` ⇒ `True`. (3) `k = 6 > 5` ⇒ `False`.

**Bài 2.** (1) `odd = 1`, `n = 9`. (2) Điều kiện `1 ≤ k ≤ 9`. (3) Mọi `k` từ 1 đến 9.

**Bài 3.** (1) `freq[ord(ch) - 97] += 1`. (2) `sum(f % 2 for f in freq)`. (3) Kiểm tra với `Counter`.

**Bài 4.** (1) Như Cách 5. (2) Thay vì `pals[0] = ch + pals[0] + ch` nhiều lần, gom các `ch` dư vào list `half`, cuối cùng `pals[0] = "".join(half) + pals[0] + "".join(reversed(half))`. (3) Kiểm tra bằng `valid_construction`.

**Bài 5.** (1) `pre[0] = 0`, `pre[i+1] = pre[i] ^ (1 << idx(s[i]))`. (2) `odd = bin(pre[r+1] ^ pre[l]).count("1")`. (3) Trả `odd <= k <= r − l + 1`.

## 19. Đáp án tham khảo

In [29]:
# Bài 1, 2
assert Solution().canConstruct("aabbc", 2) is True
assert Solution().canConstruct("aabbc", 6) is False
ks = [k for k in range(1, 15) if Solution().canConstruct("annabelle", k)]
print("k hợp lệ cho 'annabelle':", ks)
assert ks == list(range(1, 10))
print("Bài 1, 2: đúng")

k hợp lệ cho 'annabelle': [1, 2, 3, 4, 5, 6, 7, 8, 9]
Bài 1, 2: đúng


In [30]:
# Bài 3 — đáp án
def count_odd_sol(s):
    freq = [0] * 26
    for ch in s:
        freq[ord(ch) - 97] += 1
    return sum(f % 2 for f in freq)

rng = random.Random(3)
for _ in range(500):
    s = "".join(rng.choice("abcdefg") for _ in range(rng.randint(1, 30)))
    assert count_odd_sol(s) == sum(v % 2 for v in Counter(s).values())
print("Bài 3: đúng")

Bài 3: đúng


In [31]:
# Bài 4 — đáp án
def build_k_sol(s, k):
    n = len(s)
    cnt = Counter(s)
    centers = [ch for ch, v in cnt.items() if v % 2]
    if not (len(centers) <= k <= n):
        return None
    pals = list(centers)
    pairs = [ch for ch, v in cnt.items() for _ in range(v // 2)]
    need = k - len(pals)
    while need >= 2:
        ch = pairs.pop(); pals += [ch, ch]; need -= 2
    if need == 1:
        ch = pairs.pop(); pals.append(ch + ch)
    half = pairs                                   # các cặp còn dư
    pals[0] = "".join(half) + pals[0] + "".join(reversed(half))
    return pals

rng = random.Random(4)
for _ in range(1000):
    s = "".join(rng.choice("abcde") for _ in range(rng.randint(1, 25)))
    k = rng.randint(1, 30)
    res = build_k_sol(s, k)
    assert (res is not None) == Solution().canConstruct(s, k)
    if res is not None:
        assert valid_construction(s, k, res)
big = "ab" * 50000
t = time.perf_counter(); r = build_k_sol(big, 3)
print("n=1e5, k=3: độ dài các palindrome:", [len(p) for p in r], f"({time.perf_counter() - t:.3f}s)")
assert valid_construction(big, 3, r)
print("Bài 4: đúng")

n=1e5, k=3: độ dài các palindrome: [99997, 1, 2] (0.005s)
Bài 4: đúng


In [32]:
# Bài 5 — đáp án
def answer_queries_sol(s, queries):
    pre = [0]
    for ch in s:
        pre.append(pre[-1] ^ (1 << (ord(ch) - 97)))
    out = []
    for l, r, k in queries:
        odd = bin(pre[r + 1] ^ pre[l]).count("1")
        out.append(odd <= k <= r - l + 1)
    return out

rng = random.Random(5)
for _ in range(200):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 30)))
    qs = []
    for _ in range(20):
        l = rng.randrange(len(s)); r = rng.randrange(l, len(s))
        qs.append((l, r, rng.randint(1, 12)))
    got = answer_queries_sol(s, qs)
    assert got == [Solution().canConstruct(s[l:r + 1], k) for l, r, k in qs]
print(answer_queries_sol("leetcode", [(0, 7, 3), (1, 2, 1), (0, 3, 2)]))
print("Bài 5: đúng")

[False, True, True]
Bài 5: đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
def canConstruct(s, k):
    if k > len(s):
        return False
    odd = sum(v & 1 for v in Counter(s).values())   # hoặc bitmask XOR
    return odd <= k
```

**Nhớ:** "palindrome từ tập ký tự" ⇔ số chữ lẻ ≤ số palindrome; và số palindrome ≤ số ký tự.

**Checklist trước khi submit:**
- [ ] Có kiểm tra `k > len(s)` ⇒ `False`.
- [ ] Đếm chữ có tần suất **lẻ** (`v % 2 == 1`), không phải `v == 1`.
- [ ] So sánh `odd <= k`, không phải `==`.
- [ ] Bitmask dùng `^=` (lật), không dùng `|=`.
- [ ] Đã thử edge: `("a", 1)`, `("a", 2)`, `("ab", 3)`, `("aa", 2)`.
- [ ] Đúng chữ ký: `canConstruct(self, s: str, k: int) -> bool`.